# Occupied-cell equivalence 기반 pair canonicalization 실험

## 0. 읽는 방법과 학습 목표

이 Notebook은 **candidate 표현의 수**와 **실제 physical placement의 수**가 다를 때 pairwise 학습이 어떻게 달라지는지 확인하는 실행 가능한 설명서다. 위에서 아래로 Run All 하며 각 표 바로 아래의 assertion을 확인한다.

실행 후 다음 용어와 연결을 설명할 수 있어야 한다.

- **game**: 한 번의 완료된 경기. LOGO에서 한 game 전체가 held-out fold가 된다.
- **decision**: 한 piece를 어디에 둘지 선택한 한 시점.
- **candidate**: generator가 만든 placement 표현 하나. rotation/pivot 표현이 달라도 같은 칸을 차지할 수 있다.
- **selected candidate**: Human 로그가 실제로 선택한 candidate. `selectedBySource`가 label이다.
- **pair**: selected와 alternative의 16-feature 차이. pairwise LogisticRegression의 입력이다.
- **occupied cell**: placement 후 piece가 차지하는 절대 `(row, col)` 좌표.
- **equivalence class**: 같은 piece type과 같은 occupied-cell 집합을 갖는 candidate 묶음.
- **fold**: 한 game을 validation으로, 나머지 25 game을 train으로 쓰는 LOGO 한 회차.
- **leakage**: validation 정보나 사후 결과가 scaler, feature, grouping, 학습에 들어가는 오류.
- **decision-balanced weight**: candidate/class 수와 무관하게 학습 가능한 decision 하나의 symmetric sample weight 합이 1이 되도록 하는 규칙.

먼저 1~8절에서 계약과 worked example을 이해하고, 10~13절에서 한 fold의 scaler→pair→model→rank 흐름을 따라간 뒤, 14~16절에서 26-fold 결과와 판단을 읽는다.


In [ ]:
EXPERIMENT_ID = "placement-cell-equivalence-logo-v1"
LEARNING_TERMS = [
    "game", "decision", "candidate", "selected candidate", "pair",
    "occupied cell", "equivalence class", "fold", "leakage",
    "decision-balanced weight",
]

# Assertion: 이 Notebook이 설명해야 하는 최소 어휘를 실행 전에 고정한다.
assert len(LEARNING_TERMS) == 10
assert len(set(LEARNING_TERMS)) == len(LEARNING_TERMS)

# Output: 가장 먼저 학습 목표를 확인한다.
{"experimentId": EXPERIMENT_ID, "learningTerms": LEARNING_TERMS}


## 1. 실험 카드

결과를 보기 전에 질문, 가설, 비교 기준과 중단 조건을 고정한다. 결과에 맞춰 primary metric이나 gate를 바꾸지 않는다.

- 질문: 동일 physical placement의 rotation/pivot alias를 별도 negative pair로 쓰는 것이 학습과 평가를 왜곡하는가?
- 가설: occupied-cell class 단위로 pair를 만들면 selected-class 내부 alias negative와 generator multiplicity 영향을 없애고 grouped zero-difference pair가 0이 된다.
- baseline: selected candidate 대 모든 unselected candidate.
- 변경 변수: pair 생성 단위 하나(candidate → physical class).
- 고정 변수: snapshot, 26-fold LOGO, 16-feature, raw train candidate scaler, estimator, C, symmetric sample, decision weight, metric, tie-break.
- primary metric: physical-class micro MRR와 fold macro MRR.
- 중단 조건: snapshot/baseline/leakage/scaler/convergence/equivalence/determinism 계약 위반.
- 가능한 status: `blocked_by_equivalence_contract`, `unexpected_pipeline_regression`, `pair_canonicalization_supported`, `pair_canonicalization_valid_but_metric_tradeoff`.


In [ ]:
EXPERIMENT_CARD = {
    "question": "Does candidate multiplicity distort pairwise learning for one physical placement?",
    "hypothesis": "Canonical physical-class pairs remove alias negatives and multiplicity effects.",
    "baseline": "selected candidate versus every unselected candidate",
    "changedVariable": "pair construction unit only",
    "fixedVariables": [
        "snapshot", "26-fold LOGO", "16-feature schema", "shared raw-candidate scaler",
        "LogisticRegression settings", "C formula", "symmetric samples",
        "decision-balanced weights", "metrics", "Unicode code-point tie-break",
    ],
    "primaryMetrics": ["physical-class micro MRR", "physical-class macro MRR"],
    "tolerance": 1e-12,
    "possibleStatuses": [
        "blocked_by_equivalence_contract",
        "unexpected_pipeline_regression",
        "pair_canonicalization_supported",
        "pair_canonicalization_valid_but_metric_tradeoff",
    ],
}

# Assertion: primary metric과 변경 변수를 실행 전에 하나로 고정한다.
assert EXPERIMENT_CARD["changedVariable"] == "pair construction unit only"
assert EXPERIMENT_CARD["primaryMetrics"] == [
    "physical-class micro MRR", "physical-class macro MRR",
]

# Output
EXPERIMENT_CARD


## 2. 환경과 경로

수치 재현에 필요한 Python/NumPy/pandas/SciPy/scikit-learn, platform, architecture를 출력한다. dataset은 project-relative 경로로만 report에 기록하며 사용자 절대 경로는 기록하지 않는다.


In [ ]:
import hashlib
import io
import json
import math
import os
import platform
import tempfile
import warnings
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import scipy
import sklearn
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

EXPECTED_ENVIRONMENT = {
    "python": "3.12.13",
    "numpy": "2.5.1",
    "pandas": "3.0.3",
    "scipy": "1.18.0",
    "scikitLearn": "1.9.0",
}
ACTUAL_ENVIRONMENT = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "scikitLearn": sklearn.__version__,
    "system": platform.system(),
    "machine": platform.machine(),
}

# Assertion
assert {key: ACTUAL_ENVIRONMENT[key] for key in EXPECTED_ENVIRONMENT} == EXPECTED_ENVIRONMENT

# Output
display(pd.DataFrame([ACTUAL_ENVIRONMENT]))


In [ ]:
def find_repository_root(start: Path) -> Path:
    '''Find the repository without persisting its absolute path in the report.'''
    resolved = start.resolve()
    for candidate in [resolved, *resolved.parents]:
        if (candidate / "package.json").is_file() and (candidate / "AGENTS.md").is_file():
            return candidate
    raise AssertionError("REPOSITORY_ROOT_NOT_FOUND")


def sha256_file(path: Path) -> str:
    '''Return a streaming SHA-256 digest for one source file.'''
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


ROOT = find_repository_root(Path.cwd())
DATASET_RELATIVE_DIR = Path("data/datasets/human-behavior-v2-snapshot-26g-20260717")
DATASET_DIR = ROOT / DATASET_RELATIVE_DIR
SOURCE_FILES = {
    name: DATASET_DIR / name
    for name in ["manifest.json", "train.jsonl", "validation.jsonl", "rejected.jsonl"]
}
EXPECTED_SOURCE_SHA256 = {
    "manifest.json": "51027be0ebad578f2c38bb8715aa491daa8f13208e3da551a5e4a2dd08bef088",
    "train.jsonl": "85034a66373add54324a56673cb99ed5854810e4299abcaf9c195ae8cbca631e",
    "validation.jsonl": "23b832d95a2057b7b29e7062286c58c68524fa8db00f00870e6076a0a846f58a",
    "rejected.jsonl": "06ede14119014e2b040f8a4d62e00532ca6eecae9424b93e9cc882394db8a85f",
}
SOURCE_SHA256_BEFORE = {name: sha256_file(path) for name, path in SOURCE_FILES.items()}

# Assertion
assert not DATASET_RELATIVE_DIR.is_absolute() and ".." not in DATASET_RELATIVE_DIR.parts
assert all(path.is_file() for path in SOURCE_FILES.values())
assert SOURCE_SHA256_BEFORE == EXPECTED_SOURCE_SHA256, SOURCE_SHA256_BEFORE

# Output
display(pd.DataFrame([
    {"datasetPath": DATASET_RELATIVE_DIR.as_posix(), "source": name, "sha256": digest}
    for name, digest in SOURCE_SHA256_BEFORE.items()
]))


## 3. Snapshot 무결성

UTF-8, JSON object/JSONL row, schema v2, Human source, boolean label, manifest count를 검증한다. 기존 train/validation split의 gameId도 겹치면 안 된다. 이 단계가 실패하면 live log에서 재생성하지 않고 중단한다.


In [ ]:
def read_json_object(path: Path) -> dict[str, Any]:
    '''Decode one UTF-8 JSON object without replacement characters.'''
    value = json.loads(path.read_bytes().decode("utf-8", errors="strict"))
    assert isinstance(value, dict), f"JSON_OBJECT_REQUIRED:{path.name}"
    return value


def read_jsonl_frame(path: Path) -> pd.DataFrame:
    '''Validate every UTF-8 JSONL row, then load it as a DataFrame.'''
    text = path.read_bytes().decode("utf-8", errors="strict")
    lines = text.splitlines()
    assert lines, f"EMPTY_JSONL:{path.name}"
    for line_number, line in enumerate(lines, start=1):
        assert line.strip(), f"EMPTY_JSONL_LINE:{path.name}:{line_number}"
        value = json.loads(line)
        assert isinstance(value, dict), f"JSON_OBJECT_LINE_REQUIRED:{path.name}:{line_number}"
    frame = pd.read_json(io.StringIO(text), lines=True, dtype=False)
    assert len(frame) == len(lines), f"ROW_COUNT_MISMATCH:{path.name}"
    return frame


manifest = read_json_object(SOURCE_FILES["manifest.json"])
train_source = read_jsonl_frame(SOURCE_FILES["train.jsonl"])
validation_source = read_jsonl_frame(SOURCE_FILES["validation.jsonl"])
rejected_source = read_jsonl_frame(SOURCE_FILES["rejected.jsonl"])

# Assertion
assert not train_source.empty and not validation_source.empty and not rejected_source.empty

# Output
display(pd.DataFrame([
    {"file": "train.jsonl", "rows": len(train_source)},
    {"file": "validation.jsonl", "rows": len(validation_source)},
    {"file": "rejected.jsonl", "rows": len(rejected_source)},
]))


In [ ]:
REQUIRED_COLUMNS = {
    "datasetSchemaVersion", "gameId", "playerId", "pieceIndex", "source",
    "aiDifficulty", "pieceType", "candidatePlacement", "candidateResult",
    "candidateMetricsAfter", "metricDelta", "selectedBySource", "metadata",
}


def validate_candidate_rows(frame: pd.DataFrame, split_name: str) -> None:
    '''Enforce schema, Human source, and a real boolean label for every row.'''
    missing = REQUIRED_COLUMNS.difference(frame.columns)
    assert not missing, f"MISSING_COLUMNS:{split_name}:{sorted(missing)}"
    assert frame["datasetSchemaVersion"].map(
        lambda value: isinstance(value, (int, np.integer))
        and not isinstance(value, (bool, np.bool_)) and int(value) == 2
    ).all(), f"DATASET_SCHEMA_MISMATCH:{split_name}"
    assert (frame["source"] == "human").all(), f"NON_HUMAN_SOURCE:{split_name}"
    assert frame["selectedBySource"].map(
        lambda value: isinstance(value, (bool, np.bool_))
    ).all(), f"NON_BOOLEAN_SELECTED:{split_name}"


validate_candidate_rows(train_source, "train")
validate_candidate_rows(validation_source, "validation")
train_games = set(train_source["gameId"])
validation_games = set(validation_source["gameId"])
assert manifest["datasetSchemaVersion"] == 2 and manifest["sourceFilter"] == "human"
assert train_games.isdisjoint(validation_games), "ORIGINAL_SPLIT_GAME_LEAKAGE"
assert len(train_source) == manifest["trainSampleCount"]
assert len(validation_source) == manifest["validationSampleCount"]
assert len(rejected_source) == manifest["rejectedPlacementLogCount"]

train_source = train_source.copy(deep=True).assign(originalSplit="train")
validation_source = validation_source.copy(deep=True).assign(originalSplit="validation")
candidates = pd.concat([train_source, validation_source], ignore_index=True, copy=True)
assert len(candidates) == manifest["sampleCountBySource"]["human"] == 39_375
assert candidates["gameId"].nunique() == manifest["totalGameCount"] == 26

# Output
display(pd.DataFrame([{
    "games": candidates["gameId"].nunique(),
    "candidateRows": len(candidates),
    "originalSplitGameLeakage": len(train_games.intersection(validation_games)),
    "source": "human",
}]))


In [ ]:
def codepoint_key(value: Any) -> tuple[int, ...]:
    '''Create an explicit Unicode code-point sort key.'''
    return tuple(ord(character) for character in str(value))


def create_decision_key(row: pd.Series) -> str:
    '''Identify a placement decision by game, player, and piece index.'''
    assert isinstance(row["gameId"], str) and row["gameId"]
    assert isinstance(row["playerId"], str) and row["playerId"]
    assert isinstance(row["pieceIndex"], (int, np.integer))
    return f"{row['gameId']}:{row['playerId']}:{int(row['pieceIndex'])}"


def create_placement_key(row: pd.Series) -> str:
    '''Identify the raw generator representation used for canonical tie-breaks.'''
    placement = row["candidatePlacement"]
    assert placement["pieceType"] == row["pieceType"]
    return (
        f"{placement['pieceType']}:{int(placement['rotation'])}:"
        f"{int(placement['row'])}:{int(placement['col'])}"
    )


candidates["decisionKey"] = [create_decision_key(row) for _, row in candidates.iterrows()]
candidates["placementKey"] = [create_placement_key(row) for _, row in candidates.iterrows()]
decision_contract = candidates.groupby("decisionKey", sort=False).agg(
    candidateCount=("placementKey", "size"),
    placementCount=("placementKey", "nunique"),
    selectedCount=("selectedBySource", "sum"),
    gameCount=("gameId", "nunique"),
)

# Assertion
assert len(decision_contract) == manifest["acceptedPlacementLogCount"] == 1_158
assert (decision_contract["candidateCount"] >= 2).all()
assert (decision_contract["placementCount"] == decision_contract["candidateCount"]).all()
assert (decision_contract["selectedCount"] == 1).all()
assert (decision_contract["gameCount"] == 1).all()
assert int(candidates["selectedBySource"].sum()) == 1_158

# Output
display(pd.DataFrame([{
    "games": candidates["gameId"].nunique(),
    "decisions": len(decision_contract),
    "candidates": len(candidates),
    "selectedCandidates": int(candidates["selectedBySource"].sum()),
}]))


## 4. 기존 16-feature 설명

feature schema v1은 placement 직후의 결과와 보드 변화만 사용한다. 아래 표의 순서가 scaler와 coefficient의 순서다.

- `selectedBySource`는 label 식별에만 사용한다.
- `metadata.isWinner`는 사후 evaluation segmentation에만 사용한다.
- `winnerPlayerId`, heuristic score, Human input sequence는 grouping/scaler/model에 사용하지 않는다.

즉, 이 실험은 새 feature를 추가하지 않고 pair 표현만 바꾼다.


In [ ]:
FEATURE_SCHEMA_VERSION = 1
FEATURE_NAMES = [
    "candidateResult.linesCleared", "candidateResult.gameOver",
    "metricDelta.aggregateHeight", "metricDelta.maxHeight",
    "metricDelta.holes", "metricDelta.bumpiness",
    *[f"metricDelta.columnHeights[{index}]" for index in range(10)],
]
FEATURE_CATEGORIES = [
    "즉시 placement 결과", "즉시 placement 결과",
    "보드 높이", "보드 높이", "구멍", "표면 형태",
    *["열별 높이" for _ in range(10)],
]
FORBIDDEN_FEATURES = {
    "selectedBySource", "metadata.isWinner", "winnerPlayerId",
    "heuristicBaseline.score", "metadata.inputSequence", "gameId", "playerId",
}


def extract_feature_vector(row: pd.Series) -> np.ndarray:
    '''Extract exactly the ordered 16-feature allowlist from one candidate.'''
    result = row["candidateResult"]
    delta = row["metricDelta"]
    values = [
        float(result["linesCleared"]), float(result["gameOver"]),
        float(delta["aggregateHeight"]), float(delta["maxHeight"]),
        float(delta["holes"]), float(delta["bumpiness"]),
        *[float(value) for value in delta["columnHeights"]],
    ]
    vector = np.asarray(values, dtype=np.float64)
    assert vector.shape == (16,) and np.isfinite(vector).all()
    return vector


assert len(FEATURE_NAMES) == 16 and len(set(FEATURE_NAMES)) == 16
assert set(FEATURE_NAMES).isdisjoint(FORBIDDEN_FEATURES)
feature_table = pd.DataFrame({
    "index": range(16), "feature": FEATURE_NAMES, "category": FEATURE_CATEGORIES,
})

# Output
display(feature_table)


## 5. 기존 candidate pair의 작은 예제

전체 LOGO 전에 실제 held-out decision 하나로 한 pair의 생성 과정을 본다. selected와 Unicode code-point 순서의 첫 alternative를 고르고, validation game을 제외한 raw train candidate 전체에 fit한 scaler를 사용한다.

`selected - alternative`가 label 1 sample이고, 부호를 뒤집은 값이 label 0 sample이다. 이 decision의 alternative가 `n`개면 pair weight는 `1/n`, symmetric sample 양쪽은 각각 `1/(2n)`이다.


In [ ]:
candidates["featureVector"] = [extract_feature_vector(row) for _, row in candidates.iterrows()]
ordered_indices = sorted(
    candidates.index,
    key=lambda index: (
        codepoint_key(candidates.at[index, "gameId"]),
        codepoint_key(candidates.at[index, "decisionKey"]),
        codepoint_key(candidates.at[index, "placementKey"]),
    ),
)
candidates = candidates.loc[ordered_indices].reset_index(drop=True)
candidates["candidateIndex"] = np.arange(len(candidates), dtype=np.int64)
candidates["isWinner"] = candidates["metadata"].map(lambda value: bool(value["isWinner"]))
candidates["finishType"] = candidates["metadata"].map(lambda value: value["finishType"])
GAME_IDS = sorted(candidates["gameId"].unique(), key=codepoint_key)
EXAMPLE_VALIDATION_GAME = GAME_IDS[0]
example_train = candidates.loc[candidates["gameId"] != EXAMPLE_VALIDATION_GAME].copy()
example_validation = candidates.loc[candidates["gameId"] == EXAMPLE_VALIDATION_GAME].copy()
EXAMPLE_SCALER = StandardScaler().fit(np.vstack(example_train["featureVector"]))
example_train_scaled = EXAMPLE_SCALER.transform(np.vstack(example_train["featureVector"]))
example_validation_scaled = EXAMPLE_SCALER.transform(np.vstack(example_validation["featureVector"]))
EXAMPLE_VALIDATION_MAP = {
    int(index): example_validation_scaled[position]
    for position, index in enumerate(example_validation["candidateIndex"])
}

# Assertion
assert len(GAME_IDS) == 26
assert int(EXAMPLE_SCALER.n_samples_seen_) == len(example_train)
assert EXAMPLE_VALIDATION_GAME not in set(example_train["gameId"])

# Output
display(pd.DataFrame([{
    "validationGameId": EXAMPLE_VALIDATION_GAME,
    "rawTrainCandidatesInScaler": int(EXAMPLE_SCALER.n_samples_seen_),
    "validationCandidatesExcluded": len(example_validation),
}]))


In [ ]:
EXAMPLE_DECISION = sorted(example_validation["decisionKey"].unique(), key=codepoint_key)[0]
example_group = example_validation.loc[example_validation["decisionKey"] == EXAMPLE_DECISION]
selected_row = example_group.loc[example_group["selectedBySource"]].iloc[0]
alternative_rows = example_group.loc[~example_group["selectedBySource"]]
alternative_index = sorted(
    alternative_rows.index,
    key=lambda index: codepoint_key(example_validation.at[index, "placementKey"]),
)[0]
alternative_row = example_validation.loc[alternative_index]
selected_scaled = EXAMPLE_VALIDATION_MAP[int(selected_row["candidateIndex"])]
alternative_scaled = EXAMPLE_VALIDATION_MAP[int(alternative_row["candidateIndex"])]
raw_difference = selected_row["featureVector"] - alternative_row["featureVector"]
scaled_difference = selected_scaled - alternative_scaled
pair_weight = 1.0 / len(alternative_rows)

pair_example = pd.DataFrame({
    "feature": FEATURE_NAMES,
    "selectedRaw": selected_row["featureVector"],
    "alternativeRaw": alternative_row["featureVector"],
    "rawDifference": raw_difference,
    "selectedScaled": selected_scaled,
    "alternativeScaled": alternative_scaled,
    "pairDifference": scaled_difference,
})
symmetric_example = pd.DataFrame([
    {"label": 1, "differenceSign": "+", "sampleWeight": pair_weight / 2.0},
    {"label": 0, "differenceSign": "-", "sampleWeight": pair_weight / 2.0},
])

# Assertion
assert np.allclose(raw_difference, selected_row["featureVector"] - alternative_row["featureVector"])
assert np.allclose(scaled_difference, selected_scaled - alternative_scaled)
assert math.isclose(symmetric_example["sampleWeight"].sum(), pair_weight, abs_tol=1e-12)

# Output
display(pair_example)
display(symmetric_example)
display(pd.DataFrame([{
    "decisionKey": EXAMPLE_DECISION,
    "alternativeCount": len(alternative_rows),
    "pairWeight": pair_weight,
    "allPairsForDecisionWeight": pair_weight * len(alternative_rows),
}]))


## 6. Equivalence key 정의

한 decision 안에서 physical class key는 다음 두 값만 사용한다.

1. piece type
2. absolute occupied cells를 `(row, col)` Unicode와 무관한 숫자 순서로 정렬한 값

`selectedBySource`, rotation, anchor row/col, placement ID, outcome metadata, heuristic score, Human input sequence, fold metric은 key에 넣지 않는다. Python simulation이나 T-Spin/attack 재구성도 하지 않고 snapshot의 `candidatePlacement.cells`만 사용한다.


In [ ]:
EQUIVALENCE_KEY_FIELDS = ["candidatePlacement.pieceType", "candidatePlacement.cells"]
EQUIVALENCE_KEY_FORBIDDEN_FIELDS = {
    "selectedBySource", "rotation", "row", "col", "placementKey",
    "metadata.isWinner", "winnerPlayerId", "heuristicBaseline.score",
    "metadata.inputSequence", "foldMetric",
}


def create_equivalence_key(row: pd.Series) -> str:
    '''Build a label-free key from piece type and sorted absolute occupied cells.'''
    placement = row["candidatePlacement"]
    cells = sorted(
        (int(cell["row"]), int(cell["col"]))
        for cell in placement["cells"]
    )
    assert len(cells) == 4 and len(set(cells)) == 4
    encoded_cells = ";".join(f"{row_value},{col_value}" for row_value, col_value in cells)
    return f"{placement['pieceType']}|{encoded_cells}"


assert set(EQUIVALENCE_KEY_FIELDS).isdisjoint(EQUIVALENCE_KEY_FORBIDDEN_FIELDS)
candidates["equivalenceKey"] = [
    create_equivalence_key(row) for _, row in candidates.iterrows()
]

# Output
display(candidates[[
    "decisionKey", "placementKey", "selectedBySource", "equivalenceKey"
]].head(5))


## 7. 실제 duplicate decision worked example

결과가 좋아 보이는 예를 고르지 않는다. selected class multiplicity가 2 이상인 decision 중 `decisionKey` Unicode code-point 순서의 첫 번째를 사용한다. 표는 raw candidate가 다른 rotation/anchor 표현을 가져도 같은 occupied-cell key와 같은 16-feature를 공유하는지 보여준다.


In [ ]:
def build_equivalence_classes(frame: pd.DataFrame) -> pd.DataFrame:
    '''Create deterministic class membership without using the selected label in the key.'''
    records = []
    for decision_key in sorted(frame["decisionKey"].unique(), key=codepoint_key):
        decision = frame.loc[frame["decisionKey"] == decision_key]
        for class_key in sorted(decision["equivalenceKey"].unique(), key=codepoint_key):
            members = decision.loc[decision["equivalenceKey"] == class_key]
            member_indices = sorted(
                members["candidateIndex"].tolist(),
                key=lambda index: codepoint_key(frame.at[index, "placementKey"]),
            )
            records.append({
                "gameId": members["gameId"].iloc[0],
                "decisionKey": decision_key,
                "pieceType": members["pieceType"].iloc[0],
                "aiDifficulty": members["aiDifficulty"].iloc[0],
                "isWinner": bool(members["isWinner"].iloc[0]),
                "equivalenceKey": class_key,
                "representativeIndex": int(member_indices[0]),
                "memberIndices": member_indices,
                "multiplicity": int(len(member_indices)),
                "selectedCandidateCount": int(members["selectedBySource"].sum()),
                "selectedClass": bool(members["selectedBySource"].any()),
            })
    return pd.DataFrame(records)


class_frame = build_equivalence_classes(candidates)
selected_classes = class_frame.loc[class_frame["selectedClass"]]
duplicate_decisions = selected_classes.loc[
    selected_classes["multiplicity"] >= 2, "decisionKey"
].tolist()
WORKED_DECISION = sorted(duplicate_decisions, key=codepoint_key)[0]
worked_candidates = candidates.loc[candidates["decisionKey"] == WORKED_DECISION].copy()
worked_classes = class_frame.loc[class_frame["decisionKey"] == WORKED_DECISION].copy()
multiplicity_by_key = dict(zip(worked_classes["equivalenceKey"], worked_classes["multiplicity"]))
worked_candidates["classMultiplicity"] = worked_candidates["equivalenceKey"].map(multiplicity_by_key)
worked_candidates["featuresEqualWithinClass"] = worked_candidates.groupby(
    "equivalenceKey", sort=False
)["featureVector"].transform(
    lambda values: len({tuple(value) for value in values}) == 1
)

# Assertion
assert class_frame.loc[
    (class_frame["decisionKey"] == WORKED_DECISION) & class_frame["selectedClass"],
    "multiplicity",
].iloc[0] >= 2

# Output
display(worked_candidates[[
    "placementKey", "selectedBySource", "equivalenceKey",
    "classMultiplicity", "featuresEqualWithinClass",
]].assign(
    rotation=worked_candidates["candidatePlacement"].map(lambda value: value["rotation"]),
    anchorRow=worked_candidates["candidatePlacement"].map(lambda value: value["row"]),
    anchorCol=worked_candidates["candidatePlacement"].map(lambda value: value["col"]),
).sort_values("placementKey", key=lambda column: column.map(codepoint_key)))


In [ ]:
worked_candidate_count = len(worked_candidates)
worked_class_count = len(worked_classes)
worked_selected_class = worked_classes.loc[worked_classes["selectedClass"]].iloc[0]
worked_selected_alias_count = int(worked_selected_class["multiplicity"] - 1)
worked_baseline_pairs = worked_candidate_count - 1
worked_grouped_pairs = worked_class_count - 1
worked_flow = pd.DataFrame([
    {"step": "원본 candidate 수", "value": worked_candidate_count},
    {"step": "equivalence class 수", "value": worked_class_count},
    {"step": "selected class 수", "value": 1},
    {"step": "selected class 내부 alias 수", "value": worked_selected_alias_count},
    {"step": "baseline pair 수", "value": worked_baseline_pairs},
    {"step": "grouped pair 수", "value": worked_grouped_pairs},
    {"step": "baseline decision weight 합", "value": 1.0},
    {"step": "grouped decision weight 합", "value": 1.0 if worked_grouped_pairs else 0.0},
])

# Assertion
assert worked_baseline_pairs - worked_grouped_pairs == (
    worked_candidate_count - worked_class_count
)
assert worked_selected_alias_count >= 1

# Output: candidate → class → pair → weight 재분배 흐름
display(pd.DataFrame([{
    "baselinePairWeight": 1.0 / worked_baseline_pairs,
    "groupedPairWeight": 1.0 / worked_grouped_pairs,
    "baselineWeightSum": worked_baseline_pairs / worked_baseline_pairs,
    "groupedWeightSum": worked_grouped_pairs / worked_grouped_pairs,
}]))
display(worked_flow)


## 8. 전체 equivalence contract 검증

모든 class에서 occupied cells, 16-feature, `linesCleared`, `gameOver`, 저장된 `candidateMetricsAfter`와 `metricDelta`가 같아야 한다. rotation/pivot 표현, anchor, ID 차이는 허용한다.

각 decision에는 selected candidate와 selected class가 정확히 하나여야 한다. selected class 내부의 unselected alias는 허용하지만, class key 생성에는 label이 필요하지 않다. 위반 시 모델을 학습하지 않고 `blocked_by_equivalence_contract`로 중단한다.


In [ ]:
def stable_json(value: Any) -> str:
    '''Encode nested snapshot values for exact within-class comparisons.'''
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))


def validate_equivalence_contract(
    candidate_frame: pd.DataFrame,
    classes: pd.DataFrame,
) -> list[str]:
    '''Return readable contract violations; an empty list permits training.'''
    violations = []
    for _, class_row in classes.iterrows():
        members = candidate_frame.loc[class_row["memberIndices"]]
        feature_values = np.vstack(members["featureVector"])
        if not np.array_equal(feature_values, np.repeat(feature_values[:1], len(members), axis=0)):
            violations.append(f"FEATURE_MISMATCH:{class_row['decisionKey']}:{class_row['equivalenceKey']}")
        for field in ["candidateResult", "candidateMetricsAfter", "metricDelta"]:
            if members[field].map(stable_json).nunique() != 1:
                violations.append(f"IMMEDIATE_RESULT_MISMATCH:{field}:{class_row['decisionKey']}")
        if members["equivalenceKey"].nunique() != 1:
            violations.append(f"OCCUPIED_CELL_MISMATCH:{class_row['decisionKey']}")
    selected_candidate_counts = candidate_frame.groupby("decisionKey")["selectedBySource"].sum()
    selected_class_counts = classes.groupby("decisionKey")["selectedClass"].sum()
    if not (selected_candidate_counts == 1).all():
        violations.append("SELECTED_CANDIDATE_CONTRACT")
    if not (selected_class_counts == 1).all():
        violations.append("SELECTED_CLASS_CONTRACT")
    return violations


equivalence_violations = validate_equivalence_contract(candidates, class_frame)
if equivalence_violations:
    raise AssertionError({
        "status": "blocked_by_equivalence_contract",
        "violations": equivalence_violations[:20],
    })

# Assertion
assert not equivalence_violations
assert len(class_frame) <= len(candidates)

# Output
display(pd.DataFrame([{
    "status": "equivalence_contract_passed",
    "checkedClasses": len(class_frame),
    "violations": len(equivalence_violations),
}]))


## 9. 전체 class 압축 통계

candidate 수와 physical class 수의 차이는 generator 표현 중복량이다. piece, difficulty, winner/loss별로 압축을 보되, `isWinner`는 segmentation일 뿐 feature가 아니다. 이 snapshot의 Human 승리 game은 3개뿐이므로 승패별 수치는 성능 결론이 아니라 진단이다.


In [ ]:
decision_classes = class_frame.groupby("decisionKey", sort=False).size().rename("classCount")
decision_candidates = candidates.groupby("decisionKey", sort=False).size().rename("candidateCount")
decision_compression = pd.concat([decision_candidates, decision_classes], axis=1)
decision_compression["removedAliasCount"] = (
    decision_compression["candidateCount"] - decision_compression["classCount"]
)
decision_meta = candidates.drop_duplicates("decisionKey").set_index("decisionKey")
decision_compression = decision_compression.join(
    decision_meta[["gameId", "pieceType", "aiDifficulty", "isWinner"]]
)
duplicate_class_count = int((class_frame["multiplicity"] > 1).sum())
duplicate_decision_count = int(
    class_frame.loc[class_frame["multiplicity"] > 1, "decisionKey"].nunique()
)
selected_multiplicity = {
    str(int(key)): int(value)
    for key, value in selected_classes["multiplicity"].value_counts().sort_index().items()
}
pairless_decision_count = int((decision_compression["classCount"] == 1).sum())

COMPRESSION_SUMMARY = {
    "candidateCount": int(len(candidates)),
    "equivalenceClassCount": int(len(class_frame)),
    "removedAliasCount": int(len(candidates) - len(class_frame)),
    "removedAliasRatio": float((len(candidates) - len(class_frame)) / len(candidates)),
    "duplicateClassCount": duplicate_class_count,
    "duplicateDecisionCount": duplicate_decision_count,
    "selectedClassMultiplicityCounts": selected_multiplicity,
    "nonSelectedDuplicateClassCount": int(
        ((class_frame["multiplicity"] > 1) & ~class_frame["selectedClass"]).sum()
    ),
    "physicalSingleChoiceDecisionCount": pairless_decision_count,
}

# Assertion
assert COMPRESSION_SUMMARY["candidateCount"] == 39_375
assert COMPRESSION_SUMMARY["equivalenceClassCount"] == 26_403
assert COMPRESSION_SUMMARY["removedAliasCount"] == 12_972
assert COMPRESSION_SUMMARY["duplicateDecisionCount"] == 670

# Output
display(pd.DataFrame([COMPRESSION_SUMMARY]))


In [ ]:
def compression_segment(frame: pd.DataFrame, field: str) -> pd.DataFrame:
    '''Summarize candidate-to-class compression for a decision-level segment.'''
    rows = []
    for value, group in frame.groupby(field, sort=False):
        rows.append({
            field: value,
            "games": int(group["gameId"].nunique()),
            "decisions": int(len(group)),
            "candidates": int(group["candidateCount"].sum()),
            "classes": int(group["classCount"].sum()),
            "removedAliases": int(group["removedAliasCount"].sum()),
        })
    return pd.DataFrame(rows)


piece_compression = compression_segment(decision_compression, "pieceType")
difficulty_compression = compression_segment(decision_compression, "aiDifficulty")
outcome_compression = compression_segment(decision_compression, "isWinner")
winner_game_count = int(decision_compression.loc[
    decision_compression["isWinner"], "gameId"
].nunique())

# Assertion
assert winner_game_count == 3, "WINNER_GAME_COUNT_CHANGED"
assert int(piece_compression["removedAliases"].sum()) == 12_972
assert int(difficulty_compression["decisions"].sum()) == 1_158

# Output
display(piece_compression)
display(difficulty_compression)
display(outcome_compression.assign(
    interpretation=outcome_compression["isWinner"].map({
        True: "3 winner games: reference only", False: "loss segment",
    })
))


## 10. LOGO와 shared scaler 설명

LOGO는 validation game 하나를 완전히 떼고 나머지 game으로 학습한다. 각 fold에서 scaler는 **raw train candidate 전체**에 정확히 한 번 fit한다. Baseline과 grouped 모델은 그 scaler의 동일한 mean/scale과 동일하게 변환된 candidate vector를 공유한다.

Grouped class에 scaler를 다시 fit하면 pair multiplicity뿐 아니라 feature 분포까지 동시에 바뀐다. 이번 실험에서는 그 변수를 섞지 않는다.


In [ ]:
def fit_shared_scaler(
    train_frame: pd.DataFrame,
    validation_frame: pd.DataFrame,
) -> tuple[StandardScaler, np.ndarray, np.ndarray]:
    '''Fit once on raw train candidates and transform both fold partitions.'''
    train_values = np.vstack(train_frame["featureVector"])
    validation_values = np.vstack(validation_frame["featureVector"])
    scaler = StandardScaler(with_mean=True, with_std=True)
    train_scaled = scaler.fit_transform(train_values)
    validation_scaled = scaler.transform(validation_values)
    assert int(scaler.n_samples_seen_) == len(train_frame)
    assert np.isfinite(train_scaled).all() and np.isfinite(validation_scaled).all()
    return scaler, train_scaled, validation_scaled


first_scaler, first_train_scaled, first_validation_scaled = fit_shared_scaler(
    example_train, example_validation
)
baseline_scaler_mean = first_scaler.mean_.copy()
grouped_scaler_mean = first_scaler.mean_.copy()
baseline_scaler_scale = first_scaler.scale_.copy()
grouped_scaler_scale = first_scaler.scale_.copy()

# Assertion: 같은 scaler 객체의 mean/scale을 두 representation이 공유한다.
assert np.array_equal(baseline_scaler_mean, grouped_scaler_mean)
assert np.array_equal(baseline_scaler_scale, grouped_scaler_scale)
assert int(first_scaler.n_samples_seen_) == len(example_train)
assert set(example_train["gameId"]).isdisjoint(set(example_validation["gameId"]))

# Output
display(pd.DataFrame([{
    "validationGameId": EXAMPLE_VALIDATION_GAME,
    "trainGames": example_train["gameId"].nunique(),
    "trainDecisions": example_train["decisionKey"].nunique(),
    "validationDecisions": example_validation["decisionKey"].nunique(),
    "rawTrainCandidatesFit": int(first_scaler.n_samples_seen_),
    "validationCandidatesExcluded": len(example_validation),
    "sharedMeanAndScale": True,
}]))


## 11. 두 pair representation 구현

Baseline은 selected candidate와 모든 unselected candidate를 비교하므로 selected class 내부 alias도 negative pair가 된다. Grouped는 selected class 대표와 각각의 고유한 non-selected class 대표를 한 번만 비교한다.

대표 feature는 class 내부 동일성 검증 후 `placementKey` Unicode code-point 순서의 첫 candidate를 사용한다. physical class가 하나뿐인 decision은 평가에는 남지만 학습할 대안 pair가 없다. 이 decision은 weight 0으로 숨기지 않고 별도 계수하며, 실제 pair가 있는 모든 decision의 symmetric sample-weight 합이 정확히 1인지 검사한다.


In [ ]:
WEIGHT_TOLERANCE = 1e-12


def scaled_index_map(frame: pd.DataFrame, scaled: np.ndarray) -> dict[int, np.ndarray]:
    '''Map stable candidate indices to one fold's shared scaled vectors.'''
    return {
        int(index): scaled[position]
        for position, index in enumerate(frame["candidateIndex"].tolist())
    }


def create_baseline_pairs(
    train_frame: pd.DataFrame,
    scaled_by_index: dict[int, np.ndarray],
) -> pd.DataFrame:
    '''Pair the selected raw candidate with every unselected raw candidate.'''
    records = []
    for decision_key in sorted(train_frame["decisionKey"].unique(), key=codepoint_key):
        group = train_frame.loc[train_frame["decisionKey"] == decision_key]
        selected = group.loc[group["selectedBySource"]]
        alternatives = group.loc[~group["selectedBySource"]]
        assert len(selected) == 1 and not alternatives.empty
        selected_index = int(selected["candidateIndex"].iloc[0])
        pair_weight = 1.0 / len(alternatives)
        for alternative_index in alternatives["candidateIndex"]:
            records.append({
                "decisionKey": decision_key,
                "selectedIndex": selected_index,
                "alternativeIndex": int(alternative_index),
                "difference": scaled_by_index[selected_index] - scaled_by_index[int(alternative_index)],
                "pairWeight": pair_weight,
            })
    return pd.DataFrame(records)


In [ ]:
def create_grouped_pairs(
    train_frame: pd.DataFrame,
    classes: pd.DataFrame,
    scaled_by_index: dict[int, np.ndarray],
) -> tuple[pd.DataFrame, list[str]]:
    '''Pair the selected class once with each unique non-selected physical class.'''
    records = []
    pairless_decisions = []
    train_decisions = set(train_frame["decisionKey"])
    subset = classes.loc[classes["decisionKey"].isin(train_decisions)]
    for decision_key in sorted(train_decisions, key=codepoint_key):
        group = subset.loc[subset["decisionKey"] == decision_key]
        selected = group.loc[group["selectedClass"]]
        alternatives = group.loc[~group["selectedClass"]]
        assert len(selected) == 1
        if alternatives.empty:
            pairless_decisions.append(decision_key)
            continue
        selected_index = int(selected["representativeIndex"].iloc[0])
        pair_weight = 1.0 / len(alternatives)
        for alternative_index in alternatives["representativeIndex"]:
            records.append({
                "decisionKey": decision_key,
                "selectedIndex": selected_index,
                "alternativeIndex": int(alternative_index),
                "difference": scaled_by_index[selected_index] - scaled_by_index[int(alternative_index)],
                "pairWeight": pair_weight,
            })
    return pd.DataFrame(records), pairless_decisions


def create_symmetric_samples(
    pairs: pd.DataFrame,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, dict[str, float]]:
    '''Transform each pair into positive/negative samples and verify actual weights.'''
    differences = np.vstack(pairs["difference"])
    pair_weights = pairs["pairWeight"].to_numpy(dtype=np.float64)
    pair_count, feature_count = differences.shape
    x_values = np.empty((pair_count * 2, feature_count), dtype=np.float64)
    labels = np.empty(pair_count * 2, dtype=np.int64)
    sample_weights = np.empty(pair_count * 2, dtype=np.float64)
    x_values[0::2], x_values[1::2] = differences, -differences
    labels[0::2], labels[1::2] = 1, 0
    sample_weights[0::2] = pair_weights / 2.0
    sample_weights[1::2] = pair_weights / 2.0
    decision_keys = np.repeat(pairs["decisionKey"].to_numpy(), 2)
    sums = pd.DataFrame({"decisionKey": decision_keys, "weight": sample_weights}).groupby(
        "decisionKey", sort=False
    )["weight"].sum()
    assert np.allclose(sums.to_numpy(), np.ones(len(sums)), rtol=0.0, atol=WEIGHT_TOLERANCE)
    assert np.allclose(sample_weights[0::2], sample_weights[1::2], rtol=0.0, atol=0.0)
    return x_values, labels, sample_weights, {
        "representedDecisionCount": int(len(sums)),
        "minimumDecisionWeight": float(sums.min()),
        "maximumDecisionWeight": float(sums.max()),
    }


In [ ]:
first_train_map = scaled_index_map(example_train, first_train_scaled)
first_baseline_pairs = create_baseline_pairs(example_train, first_train_map)
first_grouped_pairs, first_pairless_decisions = create_grouped_pairs(
    example_train, class_frame, first_train_map
)
baseline_x, baseline_y, baseline_weights, baseline_weight_check = create_symmetric_samples(
    first_baseline_pairs
)
grouped_x, grouped_y, grouped_weights, grouped_weight_check = create_symmetric_samples(
    first_grouped_pairs
)
grouped_zero_difference_pairs = int(np.all(
    np.vstack(first_grouped_pairs["difference"]) == 0.0, axis=1
).sum())
first_train_class_counts = class_frame.loc[
    class_frame["decisionKey"].isin(set(example_train["decisionKey"]))
].groupby("decisionKey").size()

# Assertion
assert grouped_zero_difference_pairs == 0
assert set(first_pairless_decisions) == set(first_train_class_counts[first_train_class_counts == 1].index)
assert baseline_weight_check["minimumDecisionWeight"] == 1.0
assert grouped_weight_check["maximumDecisionWeight"] == 1.0

# Output
display(pd.DataFrame([
    {
        "representation": "baseline candidate pairs",
        "pairs": len(first_baseline_pairs),
        "symmetricSamples": len(baseline_x),
        "representedDecisions": baseline_weight_check["representedDecisionCount"],
        "pairlessDecisions": 0,
        "zeroDifferencePairs": int(np.all(np.vstack(first_baseline_pairs["difference"]) == 0.0, axis=1).sum()),
    },
    {
        "representation": "grouped physical-class pairs",
        "pairs": len(first_grouped_pairs),
        "symmetricSamples": len(grouped_x),
        "representedDecisions": grouped_weight_check["representedDecisionCount"],
        "pairlessDecisions": len(first_pairless_decisions),
        "zeroDifferencePairs": grouped_zero_difference_pairs,
    },
]))


## 12. 모델 설정

두 representation 모두 같은 `LogisticRegression` 설정과 같은 fold train decision 수로 계산한 C를 사용한다. hyperparameter search, 새 estimator, 새 feature는 없다. convergence warning은 숨기지 않고 hard failure로 처리한다.


In [ ]:
MODEL_CONFIG = {
    "model": "LogisticRegression",
    "solver": "lbfgs",
    "fitIntercept": False,
    "classWeight": None,
    "tolerance": 1e-12,
    "maxIterations": 10000,
    "l2": 0.01,
    "cFormula": "1/(trainDecisionCount*l2)",
}


def fit_pairwise_model(
    pairs: pd.DataFrame,
    train_decision_count: int,
) -> tuple[LogisticRegression, dict[str, Any]]:
    '''Fit the fixed pairwise model and expose convergence diagnostics.'''
    x_values, labels, sample_weights, weight_check = create_symmetric_samples(pairs)
    c_value = 1.0 / (train_decision_count * MODEL_CONFIG["l2"])
    model = LogisticRegression(
        C=c_value,
        solver=MODEL_CONFIG["solver"],
        fit_intercept=MODEL_CONFIG["fitIntercept"],
        class_weight=MODEL_CONFIG["classWeight"],
        tol=MODEL_CONFIG["tolerance"],
        max_iter=MODEL_CONFIG["maxIterations"],
    )
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        model.fit(x_values, labels, sample_weight=sample_weights)
    convergence_warnings = [
        str(item.message) for item in caught
        if issubclass(item.category, ConvergenceWarning)
    ]
    unexpected_warnings = [
        f"{item.category.__name__}:{item.message}" for item in caught
        if not issubclass(item.category, ConvergenceWarning)
    ]
    assert not convergence_warnings, convergence_warnings
    assert int(model.n_iter_[0]) < MODEL_CONFIG["maxIterations"]
    assert model.coef_.shape == (1, len(FEATURE_NAMES)) and np.isfinite(model.coef_).all()
    return model, {
        "c": float(c_value), "iterations": int(model.n_iter_[0]),
        "convergenceWarnings": convergence_warnings,
        "unexpectedWarnings": unexpected_warnings, "sampleWeight": weight_check,
    }


# Output
display(pd.DataFrame([MODEL_CONFIG]))


## 13. Metric을 작은 예제로 계산

held-out decision 하나에서 각 physical class score를 내림차순으로 정렬하고, score가 같으면 equivalence key Unicode code-point 순서를 사용한다. selected class rank로 Top-1, Top-3, reciprocal rank를 계산한다. tie-aware Top-1은 selected가 최고 score tie set에 포함되는지 보는 낙관적 진단이다.


In [ ]:
def summarize_predictions(predictions: list[dict[str, Any]]) -> dict[str, Any]:
    '''Aggregate rank records without refitting on held-out data.'''
    frame = pd.DataFrame(predictions)
    ranks = frame["selectedRank"].to_numpy(dtype=np.int64)
    return {
        "decisionCount": int(len(frame)),
        "top1Count": int((ranks == 1).sum()),
        "top3Count": int((ranks <= 3).sum()),
        "tieAwareTop1Count": int(frame["selectedAtTopScore"].sum()),
        "topScoreTieDecisionCount": int((frame["topScoreTieCount"] > 1).sum()),
        "top1Accuracy": float(np.mean(ranks == 1)),
        "top3Accuracy": float(np.mean(ranks <= 3)),
        "mrr": float(np.mean(1.0 / ranks)),
        "meanSelectedRank": float(np.mean(ranks)),
        "tieAwareTop1Accuracy": float(frame["selectedAtTopScore"].mean()),
    }


def score_raw_candidates(
    validation_frame: pd.DataFrame,
    scaled_by_index: dict[int, np.ndarray],
    coefficients: np.ndarray,
) -> list[dict[str, Any]]:
    '''Rank every raw candidate with held-out scores and a canonical tie-break.'''
    predictions = []
    for decision_key in sorted(validation_frame["decisionKey"].unique(), key=codepoint_key):
        group = validation_frame.loc[validation_frame["decisionKey"] == decision_key]
        rows = [{
            "key": row["placementKey"], "selected": bool(row["selectedBySource"]),
            "score": float(scaled_by_index[int(row["candidateIndex"])] @ coefficients),
        } for _, row in group.iterrows()]
        ranked = sorted(rows, key=lambda row: (-row["score"], codepoint_key(row["key"])))
        selected = next(row for row in rows if row["selected"])
        top_score = ranked[0]["score"]
        rank = next(index + 1 for index, row in enumerate(ranked) if row["key"] == selected["key"])
        predictions.append({
            "gameId": group["gameId"].iloc[0], "decisionKey": decision_key,
            "selectedRank": rank, "selectedAtTopScore": selected["score"] == top_score,
            "topScoreTieCount": sum(row["score"] == top_score for row in rows),
        })
    return predictions


In [ ]:
def score_physical_classes(
    validation_frame: pd.DataFrame,
    classes: pd.DataFrame,
    scaled_by_index: dict[int, np.ndarray],
    coefficients: np.ndarray,
) -> tuple[list[dict[str, Any]], dict[str, list[dict[str, Any]]]]:
    '''Rank one deterministic representative per occupied-cell class.'''
    validation_decisions = set(validation_frame["decisionKey"])
    subset = classes.loc[classes["decisionKey"].isin(validation_decisions)]
    predictions, details = [], {}
    for decision_key in sorted(validation_decisions, key=codepoint_key):
        group = subset.loc[subset["decisionKey"] == decision_key]
        rows = []
        for _, row in group.iterrows():
            representative = int(row["representativeIndex"])
            rows.append({
                "key": row["equivalenceKey"], "selected": bool(row["selectedClass"]),
                "multiplicity": int(row["multiplicity"]),
                "score": float(scaled_by_index[representative] @ coefficients),
            })
        ranked = sorted(rows, key=lambda row: (-row["score"], codepoint_key(row["key"])))
        selected = next(row for row in rows if row["selected"])
        top_score = ranked[0]["score"]
        rank = next(index + 1 for index, row in enumerate(ranked) if row["key"] == selected["key"])
        for index, row in enumerate(ranked, start=1):
            row["rank"] = index
        predictions.append({
            "gameId": validation_frame.loc[
                validation_frame["decisionKey"] == decision_key, "gameId"
            ].iloc[0],
            "decisionKey": decision_key, "selectedRank": rank,
            "selectedAtTopScore": selected["score"] == top_score,
            "topScoreTieCount": sum(row["score"] == top_score for row in rows),
        })
        details[decision_key] = ranked
    return predictions, details


In [ ]:
first_train_decision_count = int(example_train["decisionKey"].nunique())
first_baseline_model, first_baseline_fit = fit_pairwise_model(
    first_baseline_pairs, first_train_decision_count
)
first_grouped_model, first_grouped_fit = fit_pairwise_model(
    first_grouped_pairs, first_train_decision_count
)
assert first_baseline_fit["c"] == first_grouped_fit["c"]
first_validation_map = scaled_index_map(example_validation, first_validation_scaled)
first_raw_predictions = score_raw_candidates(
    example_validation, first_validation_map, first_baseline_model.coef_[0]
)
first_class_predictions, first_class_details = score_physical_classes(
    example_validation, class_frame, first_validation_map, first_baseline_model.coef_[0]
)
METRIC_EXAMPLE_DECISION = sorted(first_class_details, key=codepoint_key)[0]
metric_example_rows = first_class_details[METRIC_EXAMPLE_DECISION]
metric_example_prediction = next(
    row for row in first_class_predictions if row["decisionKey"] == METRIC_EXAMPLE_DECISION
)
metric_example_rank = metric_example_prediction["selectedRank"]

# Assertion
assert sum(row["selected"] for row in metric_example_rows) == 1
assert [row["rank"] for row in metric_example_rows] == list(range(1, len(metric_example_rows) + 1))

# Output
display(pd.DataFrame(metric_example_rows))
display(pd.DataFrame([{
    "selectedClassRank": metric_example_rank,
    "top1": metric_example_rank == 1,
    "top3": metric_example_rank <= 3,
    "reciprocalRank": 1.0 / metric_example_rank,
    "tieAwareTop1": metric_example_prediction["selectedAtTopScore"],
}]))


## 14. 2×2 평가

모델의 pair representation과 평가 unit을 분리해 네 결과를 계산한다.

1. baseline model → raw candidate evaluation
2. baseline model → physical-class evaluation
3. grouped model → raw candidate evaluation
4. grouped model → physical-class evaluation

Primary comparison은 2와 4의 physical-class micro/macro MRR이다. candidate metric과 class metric은 분모와 rank 공간이 다르므로 같은 의미로 직접 비교하지 않는다.


In [ ]:
def run_logo_fold(validation_game_id: str) -> tuple[dict[str, Any], dict[str, Any]]:
    '''Run one shared-scaler fold and return only held-out predictions and diagnostics.'''
    validation_mask = candidates["gameId"] == validation_game_id
    train_fold = candidates.loc[~validation_mask].copy()
    validation_fold = candidates.loc[validation_mask].copy()
    assert set(train_fold["gameId"]).isdisjoint(set(validation_fold["gameId"]))
    scaler, train_scaled, validation_scaled = fit_shared_scaler(train_fold, validation_fold)
    train_map = scaled_index_map(train_fold, train_scaled)
    validation_map = scaled_index_map(validation_fold, validation_scaled)
    baseline_pairs = create_baseline_pairs(train_fold, train_map)
    grouped_pairs, pairless = create_grouped_pairs(train_fold, class_frame, train_map)
    grouped_zero = int(np.all(np.vstack(grouped_pairs["difference"]) == 0.0, axis=1).sum())
    assert grouped_zero == 0
    train_decisions = int(train_fold["decisionKey"].nunique())
    baseline_model, baseline_fit = fit_pairwise_model(baseline_pairs, train_decisions)
    grouped_model, grouped_fit = fit_pairwise_model(grouped_pairs, train_decisions)
    assert baseline_fit["c"] == grouped_fit["c"]
    model_coefficients = {
        "baselineModel": baseline_model.coef_[0].copy(),
        "groupedModel": grouped_model.coef_[0].copy(),
    }
    prediction_sets = {}
    fold_metrics = {}
    for model_name, coefficients in model_coefficients.items():
        raw = score_raw_candidates(validation_fold, validation_map, coefficients)
        physical, _ = score_physical_classes(validation_fold, class_frame, validation_map, coefficients)
        prediction_sets[model_name] = {"rawCandidate": raw, "physicalClass": physical}
        fold_metrics[model_name] = {
            "rawCandidate": summarize_predictions(raw),
            "physicalClass": summarize_predictions(physical),
        }
    denominator = float(np.linalg.norm(model_coefficients["baselineModel"]) * np.linalg.norm(model_coefficients["groupedModel"]))
    diagnostics = {
        "validationGameId": validation_game_id, "trainGameCount": int(train_fold["gameId"].nunique()),
        "validationDecisionCount": int(validation_fold["decisionKey"].nunique()),
        "trainDecisionCount": train_decisions, "rawTrainCandidateCount": int(len(train_fold)),
        "scalerFitCandidateCount": int(scaler.n_samples_seen_), "gameIdLeakageCount": 0,
        "sharedScaler": True, "baselinePairCount": int(len(baseline_pairs)),
        "groupedPairCount": int(len(grouped_pairs)), "removedPairCount": int(len(baseline_pairs) - len(grouped_pairs)),
        "groupedPairlessDecisionCount": int(len(pairless)), "groupedZeroDifferencePairCount": grouped_zero,
        "baselineFit": baseline_fit, "groupedFit": grouped_fit,
        "coefficientCosineSimilarity": float(model_coefficients["baselineModel"] @ model_coefficients["groupedModel"] / denominator),
        "coefficientMaximumAbsoluteDifference": float(np.max(np.abs(model_coefficients["baselineModel"] - model_coefficients["groupedModel"]))),
        "metrics": fold_metrics,
    }
    return diagnostics, prediction_sets


In [ ]:
fold_results = []
held_out = {
    model_name: {evaluation: [] for evaluation in ["rawCandidate", "physicalClass"]}
    for model_name in ["baselineModel", "groupedModel"]
}
for fold_number, validation_game_id in enumerate(GAME_IDS, start=1):
    diagnostics, prediction_sets = run_logo_fold(validation_game_id)
    diagnostics["foldNumber"] = fold_number
    fold_results.append(diagnostics)
    for model_name in held_out:
        for evaluation_name in held_out[model_name]:
            held_out[model_name][evaluation_name].extend(
                prediction_sets[model_name][evaluation_name]
            )

METRIC_NAMES = [
    "top1Accuracy", "top3Accuracy", "mrr", "meanSelectedRank", "tieAwareTop1Accuracy",
]


def aggregate_evaluation(model_name: str, evaluation_name: str) -> dict[str, Any]:
    '''Combine held-out predictions for micro and fold metrics for distributions.'''
    metric_frame = pd.DataFrame([
        fold["metrics"][model_name][evaluation_name] for fold in fold_results
    ])
    return {
        "micro": summarize_predictions(held_out[model_name][evaluation_name]),
        "macro": {name: float(metric_frame[name].mean()) for name in METRIC_NAMES},
        "median": {name: float(metric_frame[name].median()) for name in METRIC_NAMES},
        "minimum": {name: float(metric_frame[name].min()) for name in METRIC_NAMES},
        "maximum": {name: float(metric_frame[name].max()) for name in METRIC_NAMES},
    }


RESULTS_2X2 = {
    model_name: {
        evaluation_name: aggregate_evaluation(model_name, evaluation_name)
        for evaluation_name in ["rawCandidate", "physicalClass"]
    }
    for model_name in ["baselineModel", "groupedModel"]
}


In [ ]:
legacy = RESULTS_2X2["baselineModel"]["rawCandidate"]["micro"]
baseline_class = RESULTS_2X2["baselineModel"]["physicalClass"]
grouped_class = RESULTS_2X2["groupedModel"]["physicalClass"]
baseline_reproduction = {
    "canonicalTop1Count": legacy["top1Count"],
    "canonicalTop3Count": legacy["top3Count"],
    "tieAwareTop1Count": legacy["tieAwareTop1Count"],
    "mrrRounded6": round(legacy["mrr"], 6),
    "meanSelectedRankRounded6": round(legacy["meanSelectedRank"], 6),
}

# Assertion: integer count는 exact, 표시 소수는 6자리 반올림 결과로 비교한다.
assert baseline_reproduction == {
    "canonicalTop1Count": 561, "canonicalTop3Count": 886,
    "tieAwareTop1Count": 659, "mrrRounded6": 0.645290,
    "meanSelectedRankRounded6": 2.961140,
}
assert len(fold_results) == 26
assert all(fold["gameIdLeakageCount"] == 0 for fold in fold_results)
assert all(fold["sharedScaler"] for fold in fold_results)
assert all(not fold["baselineFit"]["convergenceWarnings"] for fold in fold_results)
assert all(not fold["groupedFit"]["convergenceWarnings"] for fold in fold_results)
assert all(fold["groupedZeroDifferencePairCount"] == 0 for fold in fold_results)
assert all(
    fold["scalerFitCandidateCount"] == fold["rawTrainCandidateCount"]
    for fold in fold_results
)

# Output
result_rows = []
for model_name, evaluations in RESULTS_2X2.items():
    for evaluation_name, result in evaluations.items():
        result_rows.append({
            "model": model_name, "evaluationUnit": evaluation_name,
            **result["micro"], "macroMrr": result["macro"]["mrr"],
        })
display(pd.DataFrame(result_rows))


## 15. Fold와 모델 차이 진단

Primary class MRR의 fold별 paired delta, 개선/동일/악화 수, worst fold를 본다. coefficient cosine similarity와 최대 절댓값 차이는 두 pair representation이 모델을 얼마나 바꿨는지 보여주지만 원인 자체를 증명하지는 않는다.

또한 제거된 multiplicity와 decision별 effective pair 수 변화를 보고한다. canonical tie-break는 legacy raw candidate metric에 영향을 주므로 class metric과 구분한다.


In [ ]:
TOLERANCE = EXPERIMENT_CARD["tolerance"]
paired_deltas = []
for fold in fold_results:
    baseline_mrr = fold["metrics"]["baselineModel"]["physicalClass"]["mrr"]
    grouped_mrr = fold["metrics"]["groupedModel"]["physicalClass"]["mrr"]
    delta = grouped_mrr - baseline_mrr
    direction = "improved" if delta > TOLERANCE else "worse" if delta < -TOLERANCE else "equal"
    paired_deltas.append({
        "validationGameId": fold["validationGameId"],
        "baselineClassMrr": baseline_mrr, "groupedClassMrr": grouped_mrr,
        "delta": delta, "direction": direction,
    })
delta_frame = pd.DataFrame(paired_deltas)
worst_delta = min(paired_deltas, key=lambda row: (row["delta"], codepoint_key(row["validationGameId"])))
coefficient_summary = {
    "meanCosineSimilarity": float(np.mean([fold["coefficientCosineSimilarity"] for fold in fold_results])),
    "minimumCosineSimilarity": float(np.min([fold["coefficientCosineSimilarity"] for fold in fold_results])),
    "maximumAbsoluteDifference": float(np.max([fold["coefficientMaximumAbsoluteDifference"] for fold in fold_results])),
}
full_feature_map = {
    int(row["candidateIndex"]): row["featureVector"]
    for _, row in candidates.iterrows()
}
full_baseline_pairs = create_baseline_pairs(candidates, full_feature_map)
full_grouped_pairs, full_pairless_decisions = create_grouped_pairs(
    candidates, class_frame, full_feature_map
)
_, _, _, full_baseline_weight_check = create_symmetric_samples(full_baseline_pairs)
_, _, _, full_grouped_weight_check = create_symmetric_samples(full_grouped_pairs)
full_baseline_zero = int(np.all(
    np.vstack(full_baseline_pairs["difference"]) == 0.0, axis=1
).sum())
full_grouped_zero = int(np.all(
    np.vstack(full_grouped_pairs["difference"]) == 0.0, axis=1
).sum())
PAIR_SUMMARY = {
    "baselineFullPairCount": int(len(full_baseline_pairs)),
    "groupedFullPairCount": int(len(full_grouped_pairs)),
    "removedPairMultiplicity": int(len(candidates) - len(class_frame)),
    "baselineZeroDifferencePairCount": full_baseline_zero,
    "groupedZeroDifferencePairCount": full_grouped_zero,
    "physicalSingleChoiceDecisionCount": pairless_decision_count,
}
PRIMARY_COMPARISON = {
    "baselineMicroClassMrr": baseline_class["micro"]["mrr"],
    "groupedMicroClassMrr": grouped_class["micro"]["mrr"],
    "microDelta": grouped_class["micro"]["mrr"] - baseline_class["micro"]["mrr"],
    "baselineMacroClassMrr": baseline_class["macro"]["mrr"],
    "groupedMacroClassMrr": grouped_class["macro"]["mrr"],
    "macroDelta": grouped_class["macro"]["mrr"] - baseline_class["macro"]["mrr"],
    "improvedFoldCount": int((delta_frame["direction"] == "improved").sum()),
    "equalFoldCount": int((delta_frame["direction"] == "equal").sum()),
    "worseFoldCount": int((delta_frame["direction"] == "worse").sum()),
    "worstFold": worst_delta,
}

# Assertion
assert PAIR_SUMMARY["baselineFullPairCount"] == 38_217
assert PAIR_SUMMARY["groupedFullPairCount"] == 25_245
assert PAIR_SUMMARY["removedPairMultiplicity"] == 12_972
assert PAIR_SUMMARY["baselineZeroDifferencePairCount"] == 1_010
assert PAIR_SUMMARY["groupedZeroDifferencePairCount"] == 0
assert set(full_pairless_decisions) == set(
    decision_compression.loc[decision_compression["classCount"] == 1].index
)
assert full_baseline_weight_check["representedDecisionCount"] == 1_158
assert full_grouped_weight_check["representedDecisionCount"] == 1_156
assert PRIMARY_COMPARISON["improvedFoldCount"] + PRIMARY_COMPARISON["equalFoldCount"] + PRIMARY_COMPARISON["worseFoldCount"] == 26

# Output
display(pd.DataFrame([PRIMARY_COMPARISON]))
display(delta_frame)
display(pd.DataFrame([coefficient_summary]))
display(decision_compression[["candidateCount", "classCount", "removedAliasCount"]].describe())


## 16. 관측·해석·결정

결론은 네 층으로 읽는다.

**관측:** snapshot/equivalence/leakage/shared-scaler/baseline/convergence invariant와 실제 2×2 수치.

**해석:** alias negative 제거가 pair 수와 coefficient를 어떻게 바꿨고, primary class MRR이 어떤 방향으로 움직였는지에 대한 제한된 설명.

**결정:** gate에 따라 다음 representation을 고르거나, metric tradeoff이면 scaler canonicalization과 pair-weight 효과를 다음 실험으로 분리한다.

**아직 말할 수 없는 것:** gameplay 성능, 승률, production 일반화, causal improvement. 특히 winner game은 3개뿐이므로 별도 성능 결론을 내리지 않는다.


In [ ]:
micro_supported = PRIMARY_COMPARISON["microDelta"] >= -TOLERANCE
macro_supported = PRIMARY_COMPARISON["macroDelta"] >= -TOLERANCE
if micro_supported and macro_supported:
    EXPERIMENT_STATUS = "pair_canonicalization_supported"
    RECOMMENDATION = "Use physical-class pair canonicalization in the next controlled experiment."
else:
    EXPERIMENT_STATUS = "pair_canonicalization_valid_but_metric_tradeoff"
    RECOMMENDATION = (
        "Keep the baseline representation for now; isolate scaler canonicalization and pair-weight "
        "effects in the next experiment without adding a third model here."
    )

fold_summary = {
    "foldCount": 26,
    "convergedBaselineFoldCount": int(sum(not fold["baselineFit"]["convergenceWarnings"] for fold in fold_results)),
    "convergedGroupedFoldCount": int(sum(not fold["groupedFit"]["convergenceWarnings"] for fold in fold_results)),
    "gameIdLeakageCount": int(sum(fold["gameIdLeakageCount"] for fold in fold_results)),
    "primaryDeltaCounts": {
        key: int((delta_frame["direction"] == key).sum())
        for key in ["improved", "equal", "worse"]
    },
    "worstPrimaryDeltaFold": worst_delta,
    "baselineIterations": {
        "minimum": int(min(fold["baselineFit"]["iterations"] for fold in fold_results)),
        "maximum": int(max(fold["baselineFit"]["iterations"] for fold in fold_results)),
    },
    "groupedIterations": {
        "minimum": int(min(fold["groupedFit"]["iterations"] for fold in fold_results)),
        "maximum": int(max(fold["groupedFit"]["iterations"] for fold in fold_results)),
    },
}

# Assertion
assert EXPERIMENT_STATUS in EXPERIMENT_CARD["possibleStatuses"]
assert fold_summary["convergedBaselineFoldCount"] == 26
assert fold_summary["convergedGroupedFoldCount"] == 26
assert fold_summary["gameIdLeakageCount"] == 0

# Output
display(pd.DataFrame([{
    "status": EXPERIMENT_STATUS,
    "microClassMrrDelta": PRIMARY_COMPARISON["microDelta"],
    "macroClassMrrDelta": PRIMARY_COMPARISON["macroDelta"],
    "recommendation": RECOMMENDATION,
}]))


In [ ]:
SOURCE_SHA256_AFTER = {name: sha256_file(path) for name, path in SOURCE_FILES.items()}
assert SOURCE_SHA256_AFTER == SOURCE_SHA256_BEFORE, "SOURCE_SNAPSHOT_MUTATED"

REPORT_PATH = Path(os.environ.get(
    "PLACEMENT_CELL_EQUIVALENCE_REPORT",
    "/private/tmp/placement-cell-equivalence-logo-report.json",
))
CANONICAL_RESULT_PATH = ROOT / "docs/placement-cell-equivalence-logo-result.json"
report = {
    "experimentReportSchemaVersion": 1,
    "experimentId": EXPERIMENT_ID,
    "status": EXPERIMENT_STATUS,
    "recommendation": RECOMMENDATION,
    "sourceSnapshot": {
        "relativeDirectory": DATASET_RELATIVE_DIR.as_posix(),
        "datasetSchemaVersion": int(manifest["datasetSchemaVersion"]),
        "sha256": SOURCE_SHA256_BEFORE,
        "sourceUnchanged": True,
    },
    "environment": ACTUAL_ENVIRONMENT,
    "experimentCard": EXPERIMENT_CARD,
    "featureContract": {
        "featureSchemaVersion": FEATURE_SCHEMA_VERSION,
        "featureNames": FEATURE_NAMES,
        "selectedBySourceUsage": "label_only",
        "isWinnerUsage": "evaluation_segmentation_only",
        "forbiddenLeakageInputs": sorted(FORBIDDEN_FEATURES),
    },
    "modelConfig": MODEL_CONFIG,
    "dataSummary": {
        "gameCount": 26, "decisionCount": 1_158, "candidateCount": 39_375,
        "selectedCandidateCount": 1_158, "winnerGameCount": winner_game_count,
    },
    "baselineReproduction": baseline_reproduction,
    "equivalenceClassSummary": COMPRESSION_SUMMARY,
    "compressionByPiece": piece_compression.to_dict(orient="records"),
    "compressionByDifficulty": difficulty_compression.to_dict(orient="records"),
    "compressionByOutcome": outcome_compression.to_dict(orient="records"),
    "pairSummary": PAIR_SUMMARY,
    "results2x2": RESULTS_2X2,
    "primaryComparison": PRIMARY_COMPARISON,
    "foldSummary": fold_summary,
    "coefficientSummary": coefficient_summary,
    "foldResults": fold_results,
    "contracts": {
        "equivalenceContractPassed": True, "labelFreeGrouping": True,
        "sharedRawCandidateScalerEveryFold": True, "validationExcludedFromScaler": True,
        "gameIdLeakageCount": 0, "groupedZeroDifferencePairCount": 0,
        "baselineReproduced": True, "allFoldsConverged": True,
        "winnerSegmentReferenceOnly": True, "gameplayStrengthClaimed": False,
    },
}
report_text = json.dumps(report, ensure_ascii=False, indent=2, sort_keys=True, allow_nan=False) + "\n"


In [ ]:
def same_inode(left: Path, right: Path) -> bool:
    '''Compare existing files by device and inode.'''
    left_stat, right_stat = left.stat(), right.stat()
    return (left_stat.st_dev, left_stat.st_ino) == (right_stat.st_dev, right_stat.st_ino)


def atomic_write_text(output_path: Path, text: str, protected_sources: list[Path]) -> None:
    '''Exclusive-create a random sibling staging file and atomically replace output.'''
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_real = output_path.resolve(strict=False)
    for source_path in protected_sources:
        assert output_real != source_path.resolve(strict=True), "REPORT_SOURCE_COLLISION"
        if os.path.lexists(output_path):
            assert not output_path.is_symlink(), "REPORT_OUTPUT_SYMLINK_NOT_ALLOWED"
            assert not same_inode(output_path, source_path), "REPORT_OUTPUT_INODE_COLLISION"
    descriptor, staging_name = tempfile.mkstemp(
        prefix=f".{output_path.name}.", suffix=".tmp", dir=output_path.parent,
    )
    staging_path = Path(staging_name)
    descriptor_stat = os.fstat(descriptor)
    try:
        with os.fdopen(descriptor, "w", encoding="utf-8", newline="\n") as handle:
            handle.write(text)
            handle.flush()
            os.fsync(handle.fileno())
        staging_stat = staging_path.lstat()
        assert not staging_path.is_symlink()
        assert (staging_stat.st_dev, staging_stat.st_ino) == (
            descriptor_stat.st_dev, descriptor_stat.st_ino
        )
        assert all(not same_inode(staging_path, source) for source in protected_sources)
        os.replace(staging_path, output_path)
    finally:
        if staging_path.exists():
            staging_path.unlink()


protected_sources = list(SOURCE_FILES.values())
atomic_write_text(REPORT_PATH, report_text, protected_sources)
atomic_write_text(CANONICAL_RESULT_PATH, report_text, protected_sources)
REPORT_SHA256 = hashlib.sha256(report_text.encode("utf-8")).hexdigest()

# Assertion: temp report와 canonical result는 같은 검증 payload다.
assert sha256_file(REPORT_PATH) == REPORT_SHA256
assert sha256_file(CANONICAL_RESULT_PATH) == REPORT_SHA256
assert "timestamp" not in report_text.lower()
assert str(ROOT) not in report_text

# Output
display(pd.DataFrame([{
    "status": EXPERIMENT_STATUS, "reportSha256": REPORT_SHA256,
    "sourceSnapshotUnchanged": SOURCE_SHA256_AFTER == SOURCE_SHA256_BEFORE,
    "productionModelUpdated": False, "runtimeIntegration": False,
}]))
